In [0]:
# Databricks Notebook: L2_Plata.py
# ==============================================================================
# CAPA SILVER (L2) - Limpieza, EDA y Creación de Tabla Maestra
# Basado en segmentacionv3.py (líneas 1-200)
# ==============================================================================

import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, datediff, current_date, lit

# Inicializar Spark Session
spark = SparkSession.builder.appName("PlataRFM").getOrCreate()

# Configuración de esquemas
BRONZE_SCHEMA = "bronze"
SILVER_SCHEMA = "silver"

print("="*80)
print("🔄 CAPA SILVER: Creación de Tabla Maestra")
print("="*80)

# Crear schema Silver si no existe
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")
print(f"✅ Schema {SILVER_SCHEMA} verificado")

In [0]:
# ==============================================================================
# PASO 1: CARGAR DATOS DESDE BRONZE
# ==============================================================================

print("\n📂 Paso 1: Cargando tablas desde Bronze...")

customers = spark.table(f"{BRONZE_SCHEMA}.olist_customers")
orders = spark.table(f"{BRONZE_SCHEMA}.olist_orders")
payments = spark.table(f"{BRONZE_SCHEMA}.olist_order_payments")
order_items = spark.table(f"{BRONZE_SCHEMA}.olist_order_items")
products = spark.table(f"{BRONZE_SCHEMA}.olist_products")
category_translation = spark.table(f"{BRONZE_SCHEMA}.product_category_translation")
reviews = spark.table(f"{BRONZE_SCHEMA}.olist_reviews")

print(f"   ✅ Customers: {customers.count():,} registros")
print(f"   ✅ Orders: {orders.count():,} registros")
print(f"   ✅ Payments: {payments.count():,} registros")
print(f"   ✅ Order Items: {order_items.count():,} registros")
print(f"   ✅ Products: {products.count():,} registros")
print(f"   ✅ Categories: {category_translation.count():,} registros")
print(f"   ✅ Reviews: {reviews.count():,} registros")


In [0]:



# ==============================================================================
# PASO 2: FILTRAR PEDIDOS ENTREGADOS
# ==============================================================================

print("\n🔍 Paso 2: Filtrando solo pedidos entregados...")

orders_delivered = orders.filter(col("order_status") == "delivered")

total_orders = orders.count()
delivered_orders = orders_delivered.count()
pct_delivered = (delivered_orders / total_orders) * 100

print(f"   Total pedidos: {total_orders:,}")
print(f"   Pedidos entregados: {delivered_orders:,} ({pct_delivered:.1f}%)")
print(f"   Pedidos filtrados: {total_orders - delivered_orders:,}")

In [0]:

# ==============================================================================
# PASO 3: CREAR TABLA MAESTRA (JOINS SECUENCIALES)
# ==============================================================================

print("\n🔗 Paso 3: Creando tabla maestra con joins...")

# Join 1: Customers + Orders
print("   Joining: customers → orders_delivered")
df_merged = customers.join(orders_delivered, on="customer_id", how="inner")
print(f"      Resultado: {df_merged.count():,} registros")

# Join 2: + Payments
print("   Joining: → payments")
df_merged = df_merged.join(payments, on="order_id", how="left")
print(f"      Resultado: {df_merged.count():,} registros")

# Join 3: + Order Items
print("   Joining: → order_items")
full_data = df_merged.join(order_items, on="order_id", how="inner")
print(f"      Resultado: {full_data.count():,} registros")

# Join 4: + Products
print("   Joining: → products")
full_data = full_data.join(products, on="product_id", how="left")
print(f"      Resultado: {full_data.count():,} registros")

# Join 5: + Category Translation
print("   Joining: → category_translation")
full_data = full_data.join(category_translation, on="product_category_name", how="left")
print(f"      Resultado: {full_data.count():,} registros")

# Join 6: + Reviews
print("   Joining: → reviews")
full_data = full_data.join(
    reviews.select("order_id", "review_score"), 
    on="order_id", 
    how="left"
)
print(f"      Resultado final: {full_data.count():,} registros")

In [0]:



# ==============================================================================
# PASO 4: CALCULAR VOLUMEN DEL PRODUCTO
# ==============================================================================

print("\n🧮 Paso 4: Calculando volumen del producto (cm³)...")

# Calcular volumen (largo × alto × ancho)
full_data = full_data.withColumn(
    "product_volume_cm3",
    col("product_length_cm") * col("product_height_cm") * col("product_width_cm")
)

print("   ✅ Columna 'product_volume_cm3' creada")


In [0]:

# ==============================================================================
# PASO 5: GUARDAR TABLA MAESTRA EN SILVER
# ==============================================================================

print(f"\n💾 Paso 5: Guardando tabla maestra en {SILVER_SCHEMA}.full_data...")

# Seleccionar columnas relevantes
full_data_clean = full_data.select(
    # Customer info
    "customer_unique_id",
    "customer_id",
    "customer_city",
    "customer_state",
    
    # Order info
    "order_id",
    "order_status",
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    
    # Payment info
    "payment_type",
    "payment_installments",
    "payment_value",
    
    # Item info
    "order_item_id",
    "product_id",
    "seller_id",
    "price",
    "freight_value",
    
    # Product info
    "product_category_name",
    "product_category_name_english",
    "product_weight_g",
    "product_length_cm",
    "product_height_cm",
    "product_width_cm",
    "product_volume_cm3",
    
    # Review info
    "review_score"
)

# Guardar en Silver
full_data_clean.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{SILVER_SCHEMA}.full_data")

final_count = full_data_clean.count()
print(f"   ✅ Tabla guardada: {final_count:,} registros")

In [0]:

# ==============================================================================
# PASO 6: GENERAR TABLA RFM BASE
# ==============================================================================

from pyspark.sql import functions as F

# Fecha de referencia (hoy)
max_date = full_data_clean.agg(F.max("order_purchase_timestamp")).first()[0]
hoy = max_date + pd.Timedelta(days=1)

rfm_spark = full_data_clean.groupBy("customer_unique_id").agg(
    # Métricas RFM
    F.datediff(F.lit(hoy), F.max("order_purchase_timestamp")).alias("Recencia"),
    F.countDistinct("order_id").alias("Frecuencia"),
    F.sum("payment_value").alias("Monetario"),

    # Antigüedad
    F.datediff(F.lit(hoy), F.min("order_purchase_timestamp")).alias("Antiguedad"),

    # Comportamiento
    F.countDistinct("product_category_name_english").alias("Amplitud_Categorias"),
    F.count("order_item_id").alias("Total_Articulos"),

    # Logística
    F.avg("product_weight_g").alias("Avg_Peso_g"),
    F.avg("product_volume_cm3").alias("Avg_Volumen_cm3"),

    # Financiero
    F.avg("payment_installments").alias("Avg_Installments")
)

# Rellenar nulos con 0
rfm_spark = rfm_spark.fillna(0)

# Guardar en Silver (o Gold si prefieres)
rfm_spark.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{SILVER_SCHEMA}.rfm_features")

print("✔ RFM calculado en Spark")


In [0]:
rfm_logistica=rfm_spark.toPandas()
print(f"   ✅ Tabla RFM guardada: {len(rfm_logistica):,} clientes")

In [0]:

rfm_logistica.tail(10)

In [0]:
%skip

rfm_logistica = full_data.groupby('customer_unique_id').agg(

    # Métricas RFM
    Recencia=('order_purchase_timestamp', lambda x: (hoy - x.max()).days),
    Frecuencia=('order_id', 'nunique'),
    Monetario=('payment_value', 'sum'),

    # Métricas de Antigüedad
    Antiguedad=('order_purchase_timestamp', lambda x: (hoy - x.min()).days),

    # Métricas de Comportamiento
    Amplitud_Categorias=('product_category_name_english', 'nunique'),
    Total_Articulos=('order_item_id', 'count'),

    # Métricas de Logística (Promedio)
    Avg_Peso_g=('product_weight_g', 'mean'),
    Avg_Volumen_cm3=('product_volume_cm3', 'mean'),

    Avg_Installments=('payment_installments', 'mean')
).reset_index()

rfm_logistica = rfm_logistica.fillna(0)

rfm_logistica.head(10)

In [0]:

# ==============================================================================
# PASO 7: RESUMEN FINAL
# ==============================================================================

print("\n" + "="*80)
print("✅ CAPA SILVER COMPLETADA")
print("="*80)
print(f"Tablas creadas:")
print(f"  1. {SILVER_SCHEMA}.full_data ({final_count:,} registros)")
print(f"  2. {SILVER_SCHEMA}.rfm_features ({len(rfm_logistica):,} clientes)")
print("="*80)

# Verificar tablas creadas
spark.sql(f"SHOW TABLES IN {SILVER_SCHEMA}").show()

print("\n*** PROCESO L2 (SILVER) FINALIZADO EXITOSAMENTE ***")